In [33]:
import networkx as nx
import pandas as pd
import numpy as np
from networkx.algorithms.community import girvan_newman, louvain_communities


In [34]:

transacciones = [
    ('Cuenta_A', 'Mula_1', 1200),
    ('Cuenta_B', 'Mula_1', 1500),
    ('Cuenta_C', 'Mula_1', 1100),
    ('Mula_1', 'Hub_Lavado', 3800),
    ('Hub_Lavado', 'Cuenta_Paraiso', 3700),
    ('Cuenta_D', 'Mula_2', 900),
    ('Mula_2', 'Hub_Lavado', 900),
    ('Hub_Lavado', 'Cajero_Retiro', 1000)
]



G = nx.DiGraph()

for origen, destino, monto in transacciones:
    G.add_edge(origen, destino, weight=monto)



in_degrees = nx.in_degree_centrality(G) # cuenta las aristas que apuntan al nodo
betweenness = nx.betweenness_centrality(G) # Freciuencia con la que el nodo actua como puente obligatorio
pagerank = nx.pagerank(G, alpha=0.85, weight='weight') # Sigue el flujo del dinero


df_importancia = pd.DataFrame({
    "In_Degree": in_degrees,
    "Betweenness": betweenness,
    "PageRank": pagerank
}).sort_values(by="Betweenness", ascending=False)


print("Ranking de importancia de los nodos:")
print(df_importancia.round(4))



origen = 'Cuenta_A'
destino = 'Cuenta_Paraiso'


if nx.has_path(G, origen, destino):
    camino_mas_corto = nx.shortest_path(G, source=origen, target=destino)
    longitud = nx.shortest_path_length(G, source=origen, target=destino)
    print(f"\nCamino más corto de {origen} a {destino}: {camino_mas_corto} con longitud {longitud}")
else:
    print(f"\nNo hay camino de {origen} a {destino}.")
    
    
    


Ranking de importancia de los nodos:
                In_Degree  Betweenness  PageRank
Hub_Lavado          0.250       0.2143    0.2571
Mula_1              0.375       0.1607    0.1633
Mula_2              0.125       0.0536    0.0851
Cuenta_A            0.000       0.0000    0.0460
Cuenta_B            0.000       0.0000    0.0460
Cuenta_C            0.000       0.0000    0.0460
Cuenta_Paraiso      0.125       0.0000    0.2180
Cuenta_D            0.000       0.0000    0.0460
Cajero_Retiro       0.125       0.0000    0.0925

Camino más corto de Cuenta_A a Cuenta_Paraiso: ['Cuenta_A', 'Mula_1', 'Hub_Lavado', 'Cuenta_Paraiso'] con longitud 3


In [35]:
G = nx.karate_club_graph()


semilla = 42
# Luvian community detection
comunidad_luvian = louvain_communities(G, seed=semilla)


print("\nComunidades detectadas usando el método de Louvain:")
for i, comunidad in enumerate(comunidad_luvian):
    print(f"Comunidad {i + 1}: {list(comunidad)}")





Comunidades detectadas usando el método de Louvain:
Comunidad 1: [1, 2, 3, 7, 12, 13]
Comunidad 2: [0, 4, 5, 6, 10, 11, 16, 17, 19, 21]
Comunidad 3: [24, 25, 28, 31]
Comunidad 4: [32, 33, 8, 9, 14, 15, 18, 20, 22, 23, 26, 27, 29, 30]


In [36]:
# Girvan-Newman community detection
GN = girvan_newman(G)

nivel_uno = next(GN)
nivel_dos = next(GN)
comunidades_gn = [list(comunidad) for comunidad in nivel_dos]

print(f"Total comunidades GN: {len(comunidades_gn)}")


for i, comunidad in enumerate(comunidades_gn):
    print(f"Comunidad {i + 1}: {comunidad}")


Total comunidades GN: 3
Comunidad 1: [0, 1, 3, 4, 5, 6, 7, 10, 11, 12, 13, 16, 17, 19, 21]
Comunidad 2: [32, 33, 2, 8, 14, 15, 18, 20, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Comunidad 3: [9]


In [37]:


GD = nx.DiGraph()

np.random.seed(42)


UNormales = 80
for i in range(UNormales):
    GD.add_node(f'Usuario_Normal_{i + 1}', tipo='normal')
    
comercios = ['Oxxo', 'La Patito', 'Chedraui', 'EPOK']




for com in comercios:
    GD.add_node(com, tipo='comercio')


for i in range(UNormales):
    # Salario
    GD.add_edge('Nomina_empresa', f'USR_{i:03d}', weight=np.random.randint(5_000,20_000))
    # Compras
    come = np.random.choice(comercios[:3])
    GD.add_edge(f'USR_{i:03d}', come, weight=float(np.random.randint(100, 5_000)))
    
    
    if np.random.rand() < 0.25:  # 25% de probabilidad de que un usuario normal haga una transacción con el comercio "EPOK"
        amigo = f"USR_{np.random.randint(0, UNormales):03d}"
        if amigo != f'USR_{i:03d}':  # Evitar que un usuario haga una transacción consigo mismo
            GD.add_edge(f'USR_{i:03d}', amigo, weight=float(np.random.randint(50, 500)))
    
    
    


In [38]:
victimas = ['Victima_{i}' for i in range(1, 6)]
mulas = ['Mula_a', 'Mula_b', 'Mula_c']
hub_fraude = 'Hub_Fraude'
fachadas = ["fachada_1", "fachada_2", "fachada_3"]
salida_dinero = 'Salida_Dinero'



for victima in victimas:
    GD.add_node(victima, tipo='victima')
    
for mula in mulas:
    GD.add_node(mula, tipo='mula')

for fachada in fachadas:
    GD.add_node(fachada, tipo='fachada')
    GD.add_node(salida_dinero, tipo='Retiro')
    
    
# Transacciones de las víctimas
for victima in victimas:
    mdest = np.random.choice(mulas)
    GD.add_edge(victima, mdest, weight=14_500.0)
    
for mula in mulas:
    GD.add_edge(mula, hub_fraude, weight=30_000.0)



GD.add_edge(hub_fraude, "fachada_1", weight=50_000.0)
GD.add_edge("fachada_1", "fachada_2", weight=40_000.0)
GD.add_edge("fachada_2", "fachada_3", weight=38_000.0)
GD.add_edge("fachada_3", hub_fraude, weight=35_000.0)

GD.add_edge("fachada_3", salida_dinero, weight=11_000.0)


ciclo_sopechoso = [c for c in nx.simple_cycles(GD) if len(c) > 3]


print("\nCiclos sospechosos detectados:")
for idx , ciclo in enumerate(ciclo_sopechoso):
    print(f"Ciclo {idx + 1}: {ciclo}")


Ciclos sospechosos detectados:
Ciclo 1: ['fachada_1', 'fachada_2', 'fachada_3', 'Hub_Fraude']


In [39]:
G = GD


in_degrees = nx.in_degree_centrality(G) # cuenta las aristas que apuntan al nodo
betweenness = nx.betweenness_centrality(G) # Freciuencia con la que el nodo actua como puente obligatorio
pagerank = nx.pagerank(G, alpha=0.85, weight='weight') # Sigue el flujo del dinero


df_importancia = pd.DataFrame({
    "In_Degree": in_degrees,
    "Betweenness": betweenness,
    "PageRank": pagerank
}).sort_values(by="Betweenness", ascending=False)


print("Ranking de importancia de los nodos:")
print(df_importancia.round(4))



origen = 'Cuenta_A'
destino = 'Cuenta_Paraiso'


G_transacciones = nx.DiGraph()
G_transacciones.add_weighted_edges_from(transacciones)

if origen not in G_transacciones or destino not in G_transacciones:
    print(f"\nEl origen o destino no existe en el grafo.")
elif nx.has_path(G_transacciones, origen, destino):
    camino_mas_corto = nx.shortest_path(
        G_transacciones,
        source=origen,
        target=destino
    )
    longitud = nx.shortest_path_length(
        G_transacciones,
        source=origen,
        target=destino
    )
    print(
        f"\nCamino más corto de {origen} a {destino}: "
        f"{camino_mas_corto} con longitud {longitud}"
    )
else:
    print(f"\nNo hay camino de {origen} a {destino}.")
    

Ranking de importancia de los nodos:
                   In_Degree  Betweenness  PageRank
Hub_Fraude            0.0231       0.0006    0.0351
fachada_1             0.0058       0.0005    0.0334
fachada_2             0.0058       0.0004    0.0319
fachada_3             0.0058       0.0003    0.0307
USR_061               0.0173       0.0001    0.0047
...                      ...          ...       ...
Usuario_Normal_59     0.0000       0.0000    0.0036
Usuario_Normal_60     0.0000       0.0000    0.0036
Usuario_Normal_61     0.0000       0.0000    0.0036
Usuario_Normal_62     0.0000       0.0000    0.0036
Usuario_Normal_1      0.0000       0.0000    0.0036

[174 rows x 3 columns]

Camino más corto de Cuenta_A a Cuenta_Paraiso: ['Cuenta_A', 'Mula_1', 'Hub_Lavado', 'Cuenta_Paraiso'] con longitud 3
